# Camera Hexapod Current Analysis

The purpose of this notebook is to compare the camera hexapod forces (as measured by the strut currents). LSSTCam has additional vacuum insulated pipes (VIP) that could cause additional torques.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import sys, time, os, asyncio
import numpy as np
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import pandas as pd

from astropy.time import Time, TimeDelta
from pathlib import Path
from matplotlib.backends.backend_pdf import PdfPages
from matplotlib.gridspec import GridSpec, GridSpecFromSubplotSpec

from lsst.summit.utils.efdUtils import makeEfdClient, getEfdData

In [ ]:
plot_path = Path("./plots")
plot_path.mkdir(exist_ok=True, parents=True)

efd_client = makeEfdClient()
HEXAPOD_AXES = ["X", "Y", "Z", "U", "V", "W"]
N_AXES = len(HEXAPOD_AXES)
N_STRUTS = 6

## Helper Functions

In [ ]:
def units(ax: str) -> str:
    """Returns the camera hexapod units depending on the input string"""
    return "um" if ax.strip().upper() in "XYZ" else "º"

In [ ]:
def hide_axis(ax):
    """Hide axis labels, ticks, and spines for clean layout."""
    ax.set_xticks([])
    ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_visible(False)
    ax.set_frame_on(False)

In [ ]:
def plot_hexapod(df, title, smooth_currents=False):
    """
    Plot hexapod data as a time series.
    This will show the positions and the currents.
    """
    nrows = 3
    ncols = 2

    # Create figure and the grid for the plots
    fig = plt.figure(num=title, figsize=(10, 6))
    gs_outer = GridSpec(nrows=1, ncols=2, figure=fig)

    ax_title_left = fig.add_subplot(gs_outer[0])
    ax_title_left.set_title("CamHex Positions")
    hide_axis(ax_title_left)
    
    ax_title_right = fig.add_subplot(gs_outer[1])
    ax_title_right.set_title("Motor Currents in Struts")
    hide_axis(ax_title_right)
    
    gs_left = GridSpecFromSubplotSpec(
        nrows=nrows, ncols=ncols, subplot_spec=gs_outer[0], hspace=0, wspace=0.05
    )
    gs_right = GridSpecFromSubplotSpec(
        nrows=nrows, ncols=ncols, subplot_spec=gs_outer[1], hspace=0, wspace=0.05
    )

    # Placeholders for each plot
    axes_left = [[None for _ in range(2)] for _ in range(3)]
    axes_right = [[None for _ in range(2)] for _ in range(3)]

    # Determine the time range
    time_range = (df.index[-1] - df.index[0]).total_seconds() / 60  # Convert to minutes

    # Populate the plots
    for j in range(ncols):
        for i in range(nrows):
        
            idx = i + nrows * j
            hax = HEXAPOD_AXES[idx]

            # Left plots -- Positions --
            ax = fig.add_subplot(
                gs_left[i, j],
                sharex=axes_left[0][0] if axes_left[0][0] else None,
            )
            ax.plot(df[hax], color="royalblue")
            ax.grid(":", alpha=0.25)
            ax.set_ylabel(f"{hax} [{units(hax)}]")
            ax.tick_params(axis='both', labelsize=8)
            axes_left[i][j] = ax

            # Move ylabels and yticklabels to the right for the second column
            if j == 1:
                ax.yaxis.set_label_position("right")
                ax.yaxis.tick_right()

            # Right plots -- Currents --
            ax = fig.add_subplot(
                gs_right[i, j],
                sharex=axes_right[0][0] if axes_right[0][0] else None,
            )
            ax.plot(df[f"motorCurrent{idx}"], color="darkgreen")
            ax.grid(":", alpha=0.25)
            ax.set_ylim(-5.9, 5.9)
            ax.set_ylabel(f"Strut #{idx} [A]")
            ax.tick_params(axis='both', labelsize=8)
            axes_right[i][j] = ax

            # Move ylabels and yticklabels to the right for the second column
            if j == 1:
                ax.yaxis.set_label_position("right")
                ax.yaxis.tick_right()

        # Apply different x-tick formatting based on timespan
        if time_range > 60:
            axes_left[-1][j].xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
            axes_right[-1][j].xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
        elif time_range < 5:
            axes_left[-1][j].xaxis.set_major_formatter(mdates.DateFormatter('%H:%M:%S'))
            axes_right[-1][j].xaxis.set_major_formatter(mdates.DateFormatter('%H:%M:%S'))
    
        axes_left[-1][j].set_xlabel("Time [UTC]")
        axes_right[-1][j].set_xlabel("Time [UTC]")
    
    fig.suptitle(f"{title}\nFrom {Time(df.index[0]).isot} to {Time(df.index[-1]).isot}")
    fig.autofmt_xdate()
    fig.tight_layout()
    
    return fig

In [ ]:
def query_data(start, end):
    df_position = getEfdData(
        client=efd_client,
        topic="lsst.sal.MTHexapod.application",
        columns=[f"position{i}" for i in range(N_AXES)] + ["salIndex"],
        begin=start,
        end=end,
    )
    df_position = df_position[df_position["salIndex"] == 1]
    df_position = df_position.drop(columns="salIndex")
    df_position = df_position.rename(
        columns={f"position{i}": f"{ax}" for i, ax in enumerate(HEXAPOD_AXES)}
    )
    
    df_currents = getEfdData(
        client=efd_client,
        topic="lsst.sal.MTHexapod.electrical",
        columns=[f"motorCurrent{i}" for i in range(N_STRUTS)] + ["salIndex"],
        begin=start,
        end=end,
    )
    df_currents = df_currents[df_currents["salIndex"] == 1]
    df_currents = df_currents.drop(columns="salIndex")
    df = pd.merge_asof(
        left=df_position, right=df_currents, left_index=True, right_index=True
    )
    del df_currents, df_position

    return df

## CamHex Time Series

### LSSTCam Hexapod testing with no rotations

In [ ]:
%matplotlib inline
t_start = Time("2025-02-26T17:00:00Z", scale="utc")
t_end = Time("2025-02-26T19:00:00Z", scale="utc")

df = query_data(t_start, t_end)

fig = plot_hexapod(df, title="LSSTCam Testing 2025-02-26")
plt.show()

### LSSTCam Hexapod testing with maximal rotations


In [ ]:
%matplotlib inline
t_start = Time("2025-02-27T13:00:00Z", scale="utc")
t_end = Time("2025-02-27T18:00:00Z", scale="utc")

df = query_data(t_start, t_end)

fig = plot_hexapod(df, title="LSSTCam Testing 2025-02-27")
plt.show()

### Typical ComCam observing night.

In [ ]:
%matplotlib inline
t_start = Time("2024-12-12T05:00:00Z", scale="utc")
t_end = Time("2024-12-12T07:00:00Z", scale="utc")

df = query_data(t_start, t_end)

fig = plot_hexapod(df, title="ComCam observing 2024-12-11")
plt.show()

### Blowup of LSSTCam Hexapod testing with no rotations

In [ ]:
%matplotlib inline
t_start = Time("2025-02-26T17:37:00Z", scale="utc")
t_end = Time("2025-02-26T17:39:00Z", scale="utc")

df = query_data(t_start, t_end)

fig = plot_hexapod(df, title="LSSTCam Testing 2025-02-26 Zoomed")
plt.show()

### Hexapod warm-up - LSSTCam

In [ ]:
t_start = Time("2025-02-27T22:09:10Z", scale="utc")
t_end = Time("2025-02-27T22:09:14Z", scale="utc")

df = query_data(t_start, t_end)

fig = plot_hexapod(df, title="LSSTCam Hexapod Warm-up 2025-02-27")
plt.show()

### Hexapod warm-up - ComCam

In [ ]:
t_start = Time("2024-12-06T18:06:07Z", scale="utc")
t_end = Time("2024-12-06T18:06:11Z", scale="utc")

df = query_data(t_start, t_end)

fig = plot_hexapod(df, title="ComCam Hexapod Zoom after Warm-up 2024-12-06")
plt.show()

## Now make a pdf file with multiple time stamps

In [ ]:
starts = [
    "2024-12-08T05:00:00Z",
    "2024-12-12T05:00:00Z",
    "2025-02-26T15:00:00Z",
    "2025-02-27T13:00:00Z",
    "2025-02-28T00:02:00Z",
    "2025-02-28T17:38:00Z",
    "2025-02-28T17:57:00Z",
    "2025-03-01T13:35:00Z",
]

ends = [
    "2024-12-08T10:00:00Z",
    "2024-12-12T07:00:00Z",
    "2025-02-26T19:00:00Z",
    "2025-02-27T18:00:00Z",
    "2025-02-28T01:39:00Z",
    "2025-02-28T17:49:00Z",
    "2025-02-28T18:03:00Z",
    "2025-03-01T15:55:00Z",
]

labels = [
    "ComCam",
    "ComCam",
    "LSSTCam_Test_1",
    "LSSTCam_Test_2",
    "LSSTCam_Test_3",
    "LSSTCam_Test_4",
    "LSSTCam_Test_5",
    "LSSTCam_Test_6",
]

pdf = PdfPages(f"{plot_dir}/SITCOM-1883_12Mar25.pdf")

for i in range(len(starts)):
    
    date = starts[i].split("T")[0]
    t_start = Time(starts[i], scale="utc")
    t_end = Time(ends[i], scale="utc")

    df = query_data(t_start, t_end)
    fig = plot_hexapod(df, title=labels[i])

    print("Hexapods", labels[i], date, len(camhex), len(camhex_currents))
    pdf.savefig(fig)
    plt.clf()
    
pdf.close()
print(f"Plots completed and PDF closed")